In [ ]:
import glob
import os

import cmasher as cmr
import ipywidgets
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mpl_toolkits.basemap import Basemap
from scipy.interpolate import griddata

In [ ]:
PATH_DATA = os.path.join("..", "data")

PATH_BOOTSTRAP = os.path.join(
    PATH_DATA, "models", "mev_nn", "final_ensemble", "results_bootstrap_null"
)

PATH_OUTPUT = os.path.join(PATH_DATA, "significance_plots")
os.makedirs(PATH_OUTPUT, exist_ok=True)


cmap_name = "plasma"

In [ ]:
## Select the file for the desired return level
FILES_BOOTSTRAP = glob.glob(os.path.join(PATH_BOOTSTRAP, "*.csv"))
FILE_NAMES_BOOTSTRAP = sorted(
    [os.path.basename(x) for x in glob.glob(os.path.join(PATH_BOOTSTRAP, "*.csv"))]
)

csv_file_bootstrap = ipywidgets.Select(
    options=FILE_NAMES_BOOTSTRAP,
    value=FILE_NAMES_BOOTSTRAP[0] if FILE_NAMES_BOOTSTRAP else None,
    description="Return period:",
    disabled=False,
)
csv_file_bootstrap

In [ ]:
pvalues_data = pd.read_csv(os.path.join(PATH_BOOTSTRAP, csv_file_bootstrap.value))
pvalues_data.head()

## Plot p-values on Austria map

In [ ]:
# Plot raw p-values
cmap = cmr.get_sub_cmap(cmap_name, 0.05, 0.9)
cmap.set_extremes(over=plt.colormaps.get_cmap(cmap_name)(1.0))

plt.rcParams.update({'font.size': 18})
fig = plt.figure(figsize=(15, 10))

# Initialize the Basemap for Austria
m = Basemap(
    projection="lcc",
    resolution="f",
    lat_0=47.7,
    lon_0=13.3,
    width=6e5,
    height=3.35e5,
)
m.drawmapboundary()
m.drawcountries(color="black", linewidth=2)

# Plot raw p-values
m.scatter(
    pvalues_data["lon"],
    pvalues_data["lat"],
    c=pvalues_data["p_value_raw"],
    cmap=cmap,
    marker=",",
    s=0.7,
    latlon=True,
    vmin=0,
    vmax=1,
)

plt.title("Raw p-values")
plt.colorbar(label="p-value", fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
# Plot local_median with hatching for non-significant areas
color_country_boundary = "black"
vmin = 0
vmax = 5
factor = 1/10

cmap = cmr.get_sub_cmap(cmap_name, 0.05, 0.9)
cmap.set_extremes(over=plt.colormaps.get_cmap(cmap_name)(1.0))

plt.rcParams.update({'font.size': 18})
plt.rcParams.update({'hatch.color': 'black'})
plt.rcParams['hatch.linewidth'] = 1
fig = plt.figure(figsize=(15, 10))

# Initialize the Basemap for Austria
m = Basemap(
    projection="lcc",
    resolution="f",
    lat_0=47.7,
    lon_0=13.3,
    width=6e5,
    height=3.35e5,
)
m.drawmapboundary()
m.drawcountries(color=color_country_boundary, linewidth=2)

# Add latitude and longitude grid lines with labels
parallels = np.arange(46, 50, 1)  # Adjust range as needed
m.drawparallels(parallels, color='white', dashes=[3, 3], labels=[1, 0, 0, 0], linewidth=1.5)

meridians = np.arange(9, 18, 1)  # Adjust range as needed
m.drawmeridians(meridians, color='white', dashes=[3, 3], labels=[0, 0, 0, 1], linewidth=1.5)

# Plot local_median with color
scatter = m.scatter(
    pvalues_data["lon"],
    pvalues_data["lat"],
    c=pvalues_data["local_median"] * factor,
    cmap=cmap,
    marker=",",
    s=0.7,
    latlon=True,
    vmin=vmin,
    vmax=vmax
)

# Create grid for hatching overlay
lon = pvalues_data["lon"].values
lat = pvalues_data["lat"].values
is_sig = pvalues_data["is_significant_raw"].values

# Convert data points to projected coordinates
x_data, y_data = m(lon, lat)

# Create a regular grid in projected (lcc) space
x_min, x_max = x_data.min(), x_data.max()
y_min, y_max = y_data.min(), y_data.max()
grid_x = np.linspace(x_min, x_max, 1000)
grid_y = np.linspace(y_min, y_max, 1000)
grid_x_mesh, grid_y_mesh = np.meshgrid(grid_x, grid_y)

# Interpolate significance values in projected space
grid_sig = griddata((x_data, y_data), is_sig, (grid_x_mesh, grid_y_mesh), method='nearest')

# Convert grid back to lat/lon for use with latlon=True
grid_lon_mesh, grid_lat_mesh = m(grid_x_mesh, grid_y_mesh, inverse=True)

# Add fine-grained hatching for non-significant areas (where is_significant_raw == 0)
m.contourf(grid_lon_mesh, grid_lat_mesh, grid_sig, alpha=0, hatches=['...'], latlon=True, levels=[0, 0.5])

# Save figure
filename = os.path.splitext(csv_file_bootstrap.value)[0]
plt.savefig(os.path.join(PATH_OUTPUT, f"{filename}_local_median_hatched.png"), bbox_inches="tight")

plt.show()
plt.close()

# Save colormap separately
plt.rcParams.update({'font.size': 30})
fig = plt.figure(figsize=(3, 8))
ax1 = fig.add_axes([0.05, 0.80, 0.2, 0.9])

cb1 = mpl.colorbar.ColorbarBase(
    ax1,
    cmap=cmap,
    extend='max',
    norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax),
    orientation='vertical'
)

cb1.set_label('hailstone size [cm]')

plt.savefig(os.path.join(PATH_OUTPUT, f"{filename}_local_median_hatched_cmap.png"), bbox_inches="tight")
plt.savefig(os.path.join(PATH_OUTPUT, f"{filename}_local_median_hatched_cmap.pdf"), bbox_inches="tight")

plt.show()
plt.close()

In [ ]:
# 3D plot with local_median as Z-axis and global_median plane
%matplotlib widget
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({'font.size': 14})
fig = plt.figure(figsize=(16, 12))
ax = fig.add_subplot(111, projection='3d')

# Get data
lon = pvalues_data["lon"].values
lat = pvalues_data["lat"].values
local_median = pvalues_data["local_median"].values
global_median = pvalues_data["global_median"].values[0]  # Should be constant
is_significant = pvalues_data["is_significant_raw"].values

# Create 3D scatter plot with significance coloring
scatter = ax.scatter(
    lon,
    lat,
    local_median,
    c=is_significant,
    cmap='RdYlGn',
    marker='o',
    s=1.5,
    alpha=0.6,
    vmin=0,
    vmax=1
)

# Add horizontal plane at global_median level
lon_range = np.linspace(lon.min(), lon.max(), 20)
lat_range = np.linspace(lat.min(), lat.max(), 20)
lon_mesh, lat_mesh = np.meshgrid(lon_range, lat_range)
z_plane = np.full_like(lon_mesh, global_median)

ax.plot_surface(
    lon_mesh,
    lat_mesh,
    z_plane,
    alpha=0.2,
    color='gray',
    label=f'Global Median = {global_median:.1f}'
)

# Labels and title
ax.set_xlabel('Longitude', fontsize=14, labelpad=10)
ax.set_ylabel('Latitude', fontsize=14, labelpad=10)
ax.set_zlabel('Local Median', fontsize=14, labelpad=10)
ax.set_title('3D View: Local Median with Global Median Reference Plane', fontsize=18, pad=20)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax, label="Significant", fraction=0.03, pad=0.1, ticks=[0, 1])

# Add legend for the plane
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='gray', alpha=0.2, label=f'Global Median = {global_median:.1f}')]
ax.legend(handles=legend_elements, loc='upper left', fontsize=12)

# Set viewing angle
ax.view_init(elev=25, azim=45)

plt.tight_layout()

# Save figure
filename = os.path.splitext(csv_file_bootstrap.value)[0]
plt.savefig(os.path.join(PATH_OUTPUT, f"{filename}_3d_view.png"), bbox_inches="tight")
plt.savefig(os.path.join(PATH_OUTPUT, f"{filename}_3d_view.pdf"), bbox_inches="tight")

plt.show()
plt.close()

# Save colormap separately
plt.rcParams.update({'font.size': 30})
fig = plt.figure(figsize=(3, 8))
ax1 = fig.add_axes([0.05, 0.80, 0.2, 0.9])

cb1 = mpl.colorbar.ColorbarBase(
    ax1,
    cmap=mpl.cm.RdYlGn,
    norm=mpl.colors.Normalize(vmin=0, vmax=1),
    orientation='vertical',
    ticks=[0, 1]
)

cb1.set_label('Significant')

plt.savefig(os.path.join(PATH_OUTPUT, f"{filename}_3d_view_cmap.png"), bbox_inches="tight")
plt.savefig(os.path.join(PATH_OUTPUT, f"{filename}_3d_view_cmap.pdf"), bbox_inches="tight")

plt.close()